 Classification Models

Objective

The goal of this notebook is to train, evaluate, and compare multiple classification models for predicting customer churn. We will use a consistent preprocessing pipeline and evaluate models using standard classification metrics, with a primary focus on ROC-AUC.

Importing libraries

In [11]:
# Core libraries
import numpy as np
import pandas as pd


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# Evaluation metrics
from sklearn.metrics import (
accuracy_score,
precision_score,
recall_score,
f1_score,
roc_auc_score,
roc_curve,
confusion_matrix
)


# Reproducibility
RANDOM_STATE = 42

Load Processed Data

In [12]:
df = pd.read_csv("../data/processed/Telco-Customer-Churn-processed.csv")
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


Initial Sanity Checks

In [13]:
# Dataset shape
df.shape
# Check target distribution
df['Churn'].value_counts(normalize=True)
# Check for missing values
df.isna().sum().sort_values(ascending=False).head()

SeniorCitizen     0
tenure            0
MonthlyCharges    0
TotalCharges      0
Churn             0
dtype: int64

Train-Test Split

In [14]:
X = df.drop('Churn', axis=1)
y = df['Churn']


X_train, X_test, y_train, y_test = train_test_split(
X, y,
test_size=0.2,
stratify=y,
random_state=RANDOM_STATE
)

Feature Scaling

In [15]:
scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Baseline Model – Logistic Regression

In [16]:
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)


# Predictions
y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]
# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))

Accuracy: 0.8069552874378992
Precision: 0.6583850931677019
Recall: 0.5668449197860963
F1 Score: 0.6091954022988506
ROC-AUC: 0.8415846443979436


Decision Tree Classifier

We train a simple Decision Tree to compare against the baseline.

In [17]:
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt.fit(X_train, y_train)


y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, y_proba_dt))

ROC-AUC: 0.6622968818620992


Random Forest Classifier

An ensemble model to capture non-linear relationships.

In [18]:
rf = RandomForestClassifier(
n_estimators=200,
random_state=RANDOM_STATE,
n_jobs=-1
)


rf.fit(X_train, y_train)


y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

ROC-AUC: 0.826492546952905


Model Comparison

We compare model performance using ROC-AUC.

In [19]:
model_performance = pd.DataFrame({
'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
'ROC-AUC': [
roc_auc_score(y_test, y_proba_lr),
roc_auc_score(y_test, y_proba_dt),
roc_auc_score(y_test, y_proba_rf)
]
})


model_performance.sort_values(by='ROC-AUC', ascending=False)

,Model,ROC-AUC
0,Logistic Regression,0.841585
2,Random Forest,0.826493
1,Decision Tree,0.662297


Key Takeaways

1. Logistic Regression provides a strong and interpretable baseline

2. Tree-based models capture non-linear patterns but may require tuning

3. Random Forest typically improves performance at the cost of interpretability